# Multi-Agent Communication Protocol Study — Full Version## Efficiency, Quality & Reliability across Task Domains**GR5293 Final Project — OpenAI gpt-4o-mini****Architecture:** `Planning Agent → Execution Agent → Integration Agent` (fixed)  **Protocols:** `NL` | `MARKDOWN` | `JSON` | `SHARED_MEMORY`  **Task Domains:** `GSM8K (Math)` | `SQuAD (Reading Comprehension)` | `News Analysis`  **Evaluation Dimensions:**- **Efficiency:** token consumption, response latency- **Quality:** information fidelity, LLM-as-Judge scoring (accuracy / completeness / coherence)- **Reliability:** error propagation rate, error detection rate

## 0. Install & Import

In [ ]:
!pip install openai statsmodels -q

In [ ]:
import json, time, copy, random, hashlib, os, re
from enum import Enum
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from openai import OpenAI

print('All imports OK')

## 1. API Key Setup

In [ ]:
# Option A — Colab Secrets (recommended)
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    # Option B — environment variable or paste directly
    api_key = os.environ.get('OPENAI_API_KEY', 'sk-YOUR-KEY-HERE')

client = OpenAI(api_key=api_key)
MODEL  = 'gpt-4o-mini'

# Sanity check
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Reply with just OK'}],
    max_tokens=5
)
print('API connected:', resp.choices[0].message.content.strip())

## 2. Enums — Protocols, Failure Modes, Task Domains

In [ ]:
class Protocol(str, Enum):
    NL       = 'NL'
    MARKDOWN = 'MARKDOWN'
    JSON     = 'JSON'
    SHARED   = 'SHARED'

class FailureMode(str, Enum):
    NONE           = 'NONE'
    TOOL_FAIL      = 'TOOL_FAIL'
    CONTRADICTION  = 'CONTRADICTION'
    AMBIGUITY      = 'AMBIGUITY'
    RESOURCE_LIMIT = 'RESOURCE_LIMIT'

class TaskDomain(str, Enum):
    MATH    = 'MATH'       # GSM8K
    READING = 'READING'    # SQuAD
    NEWS    = 'NEWS'       # News analysis

print('Protocols:', [p.value for p in Protocol])
print('Failures: ', [f.value for f in FailureMode])
print('Domains:  ', [d.value for d in TaskDomain])

## 3. Communication Logger

In [ ]:
@dataclass
class Message:
    run_id:       str
    protocol:     str
    failure_mode: str
    domain:       str
    sender:       str
    receiver:     str
    content:      Any
    token_count:  int   = 0
    latency_ms:   float = 0.0
    error_flag:   bool  = False
    timestamp:    float = field(default_factory=time.time)


class CommunicationLogger:
    def __init__(self):
        self.messages: List[Message] = []

    def log(self, msg: Message):
        self.messages.append(msg)

    def summary(self) -> Dict:
        n = len(self.messages)
        return {
            'total_messages':       n,
            'total_tokens':         sum(m.token_count for m in self.messages),
            'total_latency_ms':     round(sum(m.latency_ms for m in self.messages), 2),
            'error_message_count':  sum(1 for m in self.messages if m.error_flag),
            'error_propagation_rate': round(
                sum(1 for m in self.messages if m.error_flag) / max(n, 1), 3),
        }

    def to_records(self) -> List[Dict]:
        return [asdict(m) for m in self.messages]

print('CommunicationLogger OK')

## 4. LLM Wrapper — 3-Agent Architecture**Planning Agent** → decompose task into 3-5 subtasks  **Execution Agent** → carry out subtasks  **Integration Agent** → synthesize final report

In [ ]:
# System prompts for the 3-agent architecture
SYSTEM_PROMPTS = {
    'planning':    'You are a planning agent. Decompose the given task into 3-5 clear subtasks for the execution agent.',
    'execution':   'You are an execution agent. Carry out each subtask according to the plan. Provide detailed results for each subtask.',
    'integration': 'You are an integration agent. Synthesize all execution results into a coherent, accurate final report.',
}

# Protocol-specific suffixes
PROTOCOL_SUFFIX = {
    Protocol.NL:       '',
    Protocol.MARKDOWN: ' Format your response using Markdown with headings, bullet points, and tables where appropriate.',
    Protocol.JSON:     ' Structure your response as JSON with clear field names.',
    Protocol.SHARED:   '',
}


def llm_call(role: str, prompt: str, protocol: Protocol,
             temperature: float = 0.3, max_tokens: int = 300) -> Tuple[str, float, int]:
    """
    Returns (response_text, latency_ms, total_tokens).
    """
    full_prompt = prompt + PROTOCOL_SUFFIX[protocol]
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPTS[role]},
            {'role': 'user',   'content': full_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    latency_ms   = round((time.time() - t0) * 1000, 2)
    text         = resp.choices[0].message.content.strip()
    total_tokens = resp.usage.total_tokens
    return text, latency_ms, total_tokens


print('LLM wrapper OK — 3-agent architecture')

## 5. Task Data — Three Domains- **MATH** — GSM8K samples (grade-school math, ground truth available)- **READING** — SQuAD samples (reading comprehension, ground truth available)- **NEWS** — News analysis (manually prepared reference answers)

In [ ]:
# ── GSM8K Math Tasks ──────────────────────────────────────────────────────────
MATH_TASKS = [
    {
        'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
        'ground_truth': '72',
    },
    {
        'question': 'Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?',
        'ground_truth': '10',
    },
    {
        'question': 'Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to make up the remainder?',
        'ground_truth': '5',
    },
    {
        'question': 'Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?',
        'ground_truth': '42',
    },
    {
        'question': 'James writes a 3-page letter to 2 different friends twice a week. How many pages does he write a year?',
        'ground_truth': '624',
    },
]

# ── SQuAD Reading Comprehension Tasks ────────────────────────────────────────
READING_TASKS = [
    {
        'context': 'Beyoncé Giselle Knowles-Carter is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child.',
        'question': 'When did Beyoncé rise to fame?',
        'ground_truth': 'in the late 1990s',
    },
    {
        'context': 'The Amazon rainforest produces more than 20% of the world\'s oxygen. It is often referred to as the "lungs of the Earth". The Amazon basin encompasses 5.5 million square kilometers and spans across nine countries in South America.',
        'question': 'How much of the world\'s oxygen does the Amazon rainforest produce?',
        'ground_truth': 'more than 20%',
    },
    {
        'context': 'The Nobel Prize in Physics was awarded to Albert Einstein in 1921 for his discovery of the law of the photoelectric effect. This was considered a pivotal step in the development of quantum theory.',
        'question': 'Why was Einstein awarded the Nobel Prize in Physics?',
        'ground_truth': 'for his discovery of the law of the photoelectric effect',
    },
    {
        'context': 'Python was conceived in the late 1980s by Guido van Rossum at Centrum Wiskunde & Informatica (CWI) in the Netherlands as a successor to the ABC programming language. Its implementation began in December 1989.',
        'question': 'Who created Python and where?',
        'ground_truth': 'Guido van Rossum at Centrum Wiskunde & Informatica (CWI) in the Netherlands',
    },
    {
        'context': 'The Great Wall of China is a series of fortifications that were built across the historical northern borders of ancient Chinese states to protect against various nomadic groups. The total length of all sections is approximately 21,196 kilometers.',
        'question': 'What is the total length of the Great Wall of China?',
        'ground_truth': 'approximately 21,196 kilometers',
    },
]

# ── News Analysis Tasks ──────────────────────────────────────────────────────
NEWS_TASKS = [
    {
        'article': 'The Federal Reserve announced a 25 basis point rate cut on March 15, 2026, citing softening labor market data and inflation returning to the 2% target. Markets rallied immediately, with the S&P 500 gaining 1.8% on the day. Bond yields fell sharply across the curve.',
        'question': 'Analyze the Fed\'s rate decision: what was the action, why, and how did markets react?',
        'ground_truth': 'The Fed cut rates by 25bp due to softening labor data and inflation at target. S&P 500 rose 1.8%, bond yields fell.',
    },
    {
        'article': 'A magnitude 6.2 earthquake struck central Turkey on March 10, 2026, causing building collapses in two provinces. At least 45 people were reported injured. Emergency services were deployed within hours, and the government declared a state of emergency in the affected regions.',
        'question': 'Summarize the earthquake event: location, magnitude, impact, and government response.',
        'ground_truth': 'A 6.2 magnitude earthquake hit central Turkey on March 10, injuring 45+ people, collapsing buildings. Government declared emergency, deployed services.',
    },
    {
        'article': 'Tesla reported Q4 2025 earnings with revenue of $28.4B, beating estimates by 3%. However, automotive margins fell to 16.2% from 18.1% a year earlier due to price cuts. CEO Elon Musk announced plans to begin Robotaxi production in Q2 2026.',
        'question': 'Analyze Tesla\'s earnings: revenue performance, margin trend, and future plans.',
        'ground_truth': 'Tesla Q4 revenue $28.4B beat estimates by 3%. Auto margins declined from 18.1% to 16.2% due to price cuts. Robotaxi production planned for Q2 2026.',
    },
    {
        'article': 'The EU Parliament voted 412-178 to approve the AI Liability Directive, establishing strict rules for AI-caused damages. Companies deploying high-risk AI systems will bear the burden of proof in liability cases. The directive takes effect January 2027.',
        'question': 'What did the EU Parliament decide about AI liability, and what are the key provisions?',
        'ground_truth': 'EU passed AI Liability Directive (412-178 vote). High-risk AI deployers bear burden of proof. Effective January 2027.',
    },
    {
        'article': 'Global semiconductor sales reached $620B in 2025, up 15% from 2024, driven by AI chip demand. TSMC and Samsung accounted for 72% of advanced node production. Industry analysts forecast continued growth of 12-18% in 2026 as AI infrastructure spending accelerates.',
        'question': 'Summarize global semiconductor industry performance and outlook.',
        'ground_truth': 'Global semiconductor sales hit $620B in 2025 (+15% YoY), driven by AI. TSMC+Samsung hold 72% of advanced production. 2026 forecast: 12-18% growth.',
    },
]


def get_tasks(domain: TaskDomain) -> List[Dict]:
    if domain == TaskDomain.MATH:
        return MATH_TASKS
    elif domain == TaskDomain.READING:
        return READING_TASKS
    else:
        return NEWS_TASKS


print(f'Tasks loaded: MATH={len(MATH_TASKS)}, READING={len(READING_TASKS)}, NEWS={len(NEWS_TASKS)}')

## 6. Shared Memory (for SHARED protocol)

In [ ]:
class SharedMemory:
    def __init__(self):
        self._state: Dict = {}

    def write(self, agent: str, key: str, value: Any):
        self._state[key] = value

    def read(self, key: str, default=None) -> Any:
        return self._state.get(key, default)

    def overhead_tokens(self) -> int:
        return max(1, len(json.dumps(self._state, default=str)) // 4)

    def clear(self):
        self._state = {}

print('SharedMemory OK')

## 7. Agent Functions — Planning / Execution / IntegrationEach agent communicates using the designated protocol.  Error injection modifies inputs to test reliability.

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────
ERROR_WORDS_UPSTREAM = ['unavailable', 'no data', 'not found', 'empty', 'failed', 'none',
                        'error', 'missing']
ERROR_WORDS_DETECT   = ['conflict', 'contradict', 'unavailable', 'missing', 'no data',
                        'inconsistent', 'anomaly', 'warning', 'cannot', 'unable', 'insufficient',
                        'partial', 'incomplete', 'flag', 'discrepancy', 'error']
DEGRADE_WORDS        = ['unavailable', 'incomplete', 'could not', 'unable', 'insufficient',
                        'data issue', 'caution', 'caveat', 'note:', 'however', 'unclear']

def _has_any(text: str, words: List[str]) -> bool:
    t = text.lower()
    return any(w in t for w in words)

def _to_str(data: Any) -> str:
    return json.dumps(data, default=str) if isinstance(data, dict) else str(data)


def _build_task_prompt(task: Dict, domain: TaskDomain) -> str:
    """Build the user-facing task prompt based on domain type."""
    if domain == TaskDomain.MATH:
        return f"Solve this math problem step by step: {task['question']}"
    elif domain == TaskDomain.READING:
        return (f"Read the following passage and answer the question.\n"
                f"Passage: {task['context']}\n"
                f"Question: {task['question']}")
    else:  # NEWS
        return (f"Analyze the following news article and answer the question.\n"
                f"Article: {task['article']}\n"
                f"Question: {task['question']}")


def _inject_failure(prompt: str, failure_mode: FailureMode) -> str:
    """Modify the prompt to simulate a failure condition."""
    if failure_mode == FailureMode.TOOL_FAIL:
        return 'ERROR: The data source is unavailable. No information could be retrieved. Respond based on what you can infer.'
    elif failure_mode == FailureMode.CONTRADICTION:
        return prompt + '\n\nHowever, a conflicting source states the opposite conclusion. Note: there may be contradictory information above.'
    elif failure_mode == FailureMode.AMBIGUITY:
        return 'Please look into the topic mentioned somewhere and provide something useful about it.'
    elif failure_mode == FailureMode.RESOURCE_LIMIT:
        # Truncate to simulate partial data
        words = prompt.split()
        return ' '.join(words[:len(words)//2]) + ' [TRUNCATED — resource limit reached]'
    return prompt


# ── agents ────────────────────────────────────────────────────────────────────
def agent_planning(task_prompt: str, protocol: Protocol, failure_mode: FailureMode,
                   domain: TaskDomain, logger: CommunicationLogger, run_id: str) -> Tuple[str, int]:
    """Planning Agent: decompose task into subtasks."""
    if failure_mode == FailureMode.AMBIGUITY:
        task_prompt = _inject_failure(task_prompt, failure_mode)

    prompt = f'Decompose this task into 3-5 subtasks for the execution agent:\n{task_prompt}'
    text, ms, tokens = llm_call('planning', prompt, protocol)

    logger.log(Message(run_id=run_id, protocol=protocol.value, failure_mode=failure_mode.value,
                       domain=domain.value, sender='Planning', receiver='Execution',
                       content=text, token_count=tokens, latency_ms=ms))
    return text, tokens


def agent_execution(plan: str, task_prompt: str, protocol: Protocol,
                    failure_mode: FailureMode, domain: TaskDomain,
                    logger: CommunicationLogger, run_id: str) -> Tuple[str, bool, int]:
    """Execution Agent: carry out subtasks based on the plan."""
    combined = f'Plan from planning agent:\n{plan}\n\nOriginal task context:\n{task_prompt}'

    if failure_mode in (FailureMode.TOOL_FAIL, FailureMode.CONTRADICTION, FailureMode.RESOURCE_LIMIT):
        combined = _inject_failure(combined, failure_mode)

    prompt = f'Execute the following plan and provide detailed results for each subtask:\n{combined}'
    text, ms, tokens = llm_call('execution', prompt, protocol)

    error_signal = _has_any(text, ERROR_WORDS_UPSTREAM)

    logger.log(Message(run_id=run_id, protocol=protocol.value, failure_mode=failure_mode.value,
                       domain=domain.value, sender='Execution', receiver='Integration',
                       content=text, token_count=tokens, latency_ms=ms, error_flag=error_signal))
    return text, error_signal, tokens


def agent_integration(execution_result: str, protocol: Protocol,
                      failure_mode: FailureMode, domain: TaskDomain,
                      logger: CommunicationLogger, run_id: str) -> Tuple[str, bool, bool, int]:
    """Integration Agent: synthesize final report from execution results."""
    prompt = (f'Synthesize the following execution results into a coherent final report. '
              f'If you notice any data quality issues, contradictions, or missing information, '
              f'flag them explicitly.\nExecution results:\n{execution_result}')

    text, ms, tokens = llm_call('integration', prompt, protocol)

    error_detected  = _has_any(text, ERROR_WORDS_DETECT)
    upstream_error  = _has_any(execution_result, ERROR_WORDS_UPSTREAM)
    propagate       = upstream_error and not error_detected
    report_degraded = _has_any(text, DEGRADE_WORDS)

    logger.log(Message(run_id=run_id, protocol=protocol.value, failure_mode=failure_mode.value,
                       domain=domain.value, sender='Integration', receiver='Output',
                       content=text, token_count=tokens, latency_ms=ms, error_flag=propagate))
    return text, error_detected, report_degraded, tokens


print('All 3 agents OK — Planning / Execution / Integration')

## 8. LLM-as-Judge — Quality EvaluationEach report is scored on three dimensions:- **Accuracy** (1-5): correctness relative to ground truth- **Completeness** (1-5): coverage of key information- **Coherence** (1-5): logical flow and clarityEach report is judged 3 times and averaged to reduce randomness.

In [ ]:
JUDGE_PROMPT_TEMPLATE = """You are an impartial evaluator. Score the following report on three dimensions.

**Ground Truth / Reference Answer:**
{ground_truth}

**Report to Evaluate:**
{report}

Score each dimension from 1 (worst) to 5 (best):
- Accuracy: How correct is the report compared to the ground truth?
- Completeness: Does the report cover all key information from the ground truth?
- Coherence: Is the report logically structured, clear, and well-written?

Respond ONLY with a JSON object, no other text:
{{"accuracy": <int 1-5>, "completeness": <int 1-5>, "coherence": <int 1-5>}}
"""

def llm_judge(report: str, ground_truth: str, n_judges: int = 3) -> Dict[str, float]:
    """
    Run LLM-as-Judge n_judges times and average scores.
    Returns dict with accuracy, completeness, coherence (averaged).
    """
    scores = {'accuracy': [], 'completeness': [], 'coherence': []}

    for _ in range(n_judges):
        prompt = JUDGE_PROMPT_TEMPLATE.format(ground_truth=ground_truth, report=report)
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': 'You are a strict, impartial evaluator. Output only valid JSON.'},
                    {'role': 'user',   'content': prompt},
                ],
                temperature=0.1,
                max_tokens=60,
            )
            text = resp.choices[0].message.content.strip()
            # Parse JSON — handle possible markdown code fences
            text = re.sub(r'^```json\s*', '', text)
            text = re.sub(r'\s*```$', '', text)
            parsed = json.loads(text)
            for k in scores:
                scores[k].append(int(parsed.get(k, 3)))
        except Exception as e:
            # Default score on failure
            for k in scores:
                scores[k].append(3)

    return {k: round(np.mean(v), 2) for k, v in scores.items()}


# Quick test
test_score = llm_judge('The answer is 72.', 'The answer is 72.')
print('Judge test:', test_score)

## 9. Information Fidelity MetricMeasures how well key information is preserved as it passes between agents.Compares the original task content with the final report using key-fact overlap.

In [ ]:
def compute_fidelity(ground_truth: str, report: str) -> float:
    """
    Simple token-overlap fidelity score.
    Measures what fraction of ground-truth key tokens appear in the report.
    Returns a float in [0, 1].
    """
    # Extract meaningful tokens (lowercase, alphanumeric, length >= 2)
    def tokenize(text):
        return set(w.lower() for w in re.findall(r'[a-zA-Z0-9]+', text) if len(w) >= 2)

    gt_tokens = tokenize(ground_truth)
    rpt_tokens = tokenize(report)

    if not gt_tokens:
        return 1.0

    overlap = gt_tokens & rpt_tokens
    return round(len(overlap) / len(gt_tokens), 3)


# Quick test
print('Fidelity test:', compute_fidelity('The answer is 72 clips total', 'Natalia sold 72 clips in total'))

## 10. Pipeline Runner — Full Experiment

In [ ]:
@dataclass
class RunResult:
    run_id:                 str
    protocol:               str
    failure_mode:           str
    domain:                 str
    task_idx:               int
    # Efficiency
    total_tokens:           int
    total_latency_ms:       float
    total_messages:         int
    # Quality
    accuracy_score:         float
    completeness_score:     float
    coherence_score:        float
    fidelity:               float
    # Reliability
    error_count:            int
    error_propagation_rate: float
    error_detected:         bool
    report_degraded:        bool
    recovery_success:       bool
    report_snippet:         str


def run_pipeline(protocol: Protocol, failure_mode: FailureMode,
                 domain: TaskDomain, task_idx: int = 0,
                 seed: int = 0) -> RunResult:

    random.seed(seed)
    tasks = get_tasks(domain)
    task = tasks[task_idx % len(tasks)]
    task_prompt = _build_task_prompt(task, domain)
    ground_truth = task['ground_truth']

    run_id = hashlib.md5(f'{protocol}{failure_mode}{domain}{task_idx}{seed}'.encode()).hexdigest()[:8]
    logger = CommunicationLogger()
    shared = SharedMemory()

    if protocol == Protocol.SHARED:
        plan, _  = agent_planning(task_prompt, protocol, failure_mode, domain, logger, run_id)
        shared.write('Planning', 'plan', plan)

        exec_out, _, _ = agent_execution(
            shared.read('plan', ''), task_prompt, protocol, failure_mode, domain, logger, run_id)
        shared.write('Execution', 'result', exec_out)

        report, detected, degraded, _ = agent_integration(
            shared.read('result', ''), protocol, failure_mode, domain, logger, run_id)
        shared.write('Integration', 'report', report)

        extra_tokens = shared.overhead_tokens()
    else:
        plan, _  = agent_planning(task_prompt, protocol, failure_mode, domain, logger, run_id)
        exec_out, _, _ = agent_execution(plan, task_prompt, protocol, failure_mode, domain, logger, run_id)
        report, detected, degraded, _ = agent_integration(exec_out, protocol, failure_mode, domain, logger, run_id)
        extra_tokens = 0

    # ── Quality Evaluation ────────────────────────────────────────────────
    judge_scores = llm_judge(report, ground_truth, n_judges=3)
    fidelity     = compute_fidelity(ground_truth, report)

    s = logger.summary()
    return RunResult(
        run_id=run_id,
        protocol=protocol.value,
        failure_mode=failure_mode.value,
        domain=domain.value,
        task_idx=task_idx,
        total_tokens=s.get('total_tokens', 0) + extra_tokens,
        total_latency_ms=s.get('total_latency_ms', 0),
        total_messages=s.get('total_messages', 0),
        accuracy_score=judge_scores['accuracy'],
        completeness_score=judge_scores['completeness'],
        coherence_score=judge_scores['coherence'],
        fidelity=fidelity,
        error_count=s.get('error_message_count', 0),
        error_propagation_rate=s.get('error_propagation_rate', 0.0),
        error_detected=detected,
        report_degraded=degraded,
        recovery_success=detected and not degraded,
        report_snippet=report[:200],
    )


print('Pipeline runner OK — with quality evaluation')

## 11. Cost Estimator — Run Before Full Grid

In [ ]:
N_REPS       = 3   # repetitions per cell (adjust before running)
N_TASKS      = 5   # tasks per domain per cell
N_PROTOCOLS  = len(list(Protocol))       # 4
N_FAILURES   = len(list(FailureMode))    # 5
N_DOMAINS    = len(list(TaskDomain))     # 3

# Each run: 3 agent calls + 3 judge calls = 6 API calls
N_CELLS      = N_PROTOCOLS * N_FAILURES * N_DOMAINS  # 4x5x3 = 60
N_TOTAL_RUNS = N_CELLS * N_TASKS * N_REPS
API_CALLS    = N_TOTAL_RUNS * 6  # 3 agents + 3 judges

# gpt-4o-mini pricing: $0.15/1M input, $0.60/1M output
est_in  = API_CALLS * 600   # ~600 input tokens per call
est_out = API_CALLS * 200   # ~200 output tokens per call
est_usd = est_in / 1e6 * 0.15 + est_out / 1e6 * 0.60

print(f'Grid:          {N_PROTOCOLS} protocols x {N_FAILURES} failure modes x {N_DOMAINS} domains')
print(f'Tasks/cell:    {N_TASKS} tasks x {N_REPS} reps')
print(f'Total runs:    {N_TOTAL_RUNS}')
print(f'API calls:     {API_CALLS}')
print(f'Estimated cost: ${est_usd:.2f}')
print(f'\n⚠️  Adjust N_REPS and N_TASKS above to control cost.')

## 12. Run Full Experiment Grid> ⚠️ This cell makes real API calls. Check estimated cost above first.

In [ ]:
results = []
total   = N_TOTAL_RUNS
done    = 0

for protocol in Protocol:
    for failure_mode in FailureMode:
        for domain in TaskDomain:
            for task_idx in range(N_TASKS):
                for rep in range(N_REPS):
                    try:
                        r = run_pipeline(protocol, failure_mode, domain,
                                         task_idx=task_idx, seed=rep)
                        results.append(asdict(r))
                    except Exception as e:
                        print(f'  ERROR {protocol.value}/{failure_mode.value}/{domain.value}/t{task_idx}/r{rep}: {e}')
                    done += 1
                    if done % 30 == 0:
                        print(f'  {done}/{total} runs done...')

df = pd.DataFrame(results)
print(f'\nDone! {len(df)} runs collected.')
print(df.head())

## 13. Save Raw Results

In [ ]:
df.to_csv('results_raw.csv', index=False)
print(f'Saved: results_raw.csv  ({len(df)} rows)')

## 14. RQ1 — Protocol Efficiency (Baseline, no failure)

In [ ]:
baseline = df[df['failure_mode'] == 'NONE']

# ── Efficiency by Protocol ────────────────────────────────────────────────
eff = baseline.groupby('protocol').agg(
    mean_tokens   = ('total_tokens',     'mean'),
    std_tokens    = ('total_tokens',      'std'),
    mean_latency  = ('total_latency_ms', 'mean'),
    std_latency   = ('total_latency_ms',  'std'),
).round(2)

print('RQ1: Protocol Efficiency — Baseline (no failure)')
print(eff.to_string())

# ── ANOVA on tokens ──────────────────────────────────────────────────────
groups = [baseline[baseline['protocol'] == p]['total_tokens'].values
          for p in ['NL','MARKDOWN','JSON','SHARED']]
f_stat, p_val = stats.f_oneway(*groups)
print(f'\nOne-way ANOVA (tokens): F={f_stat:.3f}, p={p_val:.4f}',
      '→ Significant' if p_val < 0.05 else '→ ns')

# ── Efficiency by Protocol x Domain ──────────────────────────────────────
eff_domain = baseline.groupby(['protocol', 'domain']).agg(
    mean_tokens  = ('total_tokens',     'mean'),
    mean_latency = ('total_latency_ms', 'mean'),
).round(2)
print('\nEfficiency by Protocol x Domain:')
print(eff_domain.to_string())

## 15. RQ2 — Output Quality (LLM-as-Judge + Fidelity)New evaluation dimension: how protocol choice affects final output quality.

In [ ]:
# ── Quality by Protocol (baseline) ────────────────────────────────────────
quality = baseline.groupby('protocol').agg(
    mean_accuracy     = ('accuracy_score',     'mean'),
    mean_completeness = ('completeness_score', 'mean'),
    mean_coherence    = ('coherence_score',    'mean'),
    mean_fidelity     = ('fidelity',           'mean'),
).round(3)

print('RQ2: Output Quality — Baseline')
print(quality.to_string())

# ── ANOVA on accuracy scores ─────────────────────────────────────────────
groups_acc = [baseline[baseline['protocol'] == p]['accuracy_score'].values
              for p in ['NL','MARKDOWN','JSON','SHARED']]
f_acc, p_acc = stats.f_oneway(*groups_acc)
print(f'\nANOVA (accuracy): F={f_acc:.3f}, p={p_acc:.4f}',
      '→ Significant' if p_acc < 0.05 else '→ ns')

# ── Quality by Protocol x Domain ─────────────────────────────────────────
quality_domain = baseline.groupby(['protocol', 'domain']).agg(
    accuracy     = ('accuracy_score',     'mean'),
    completeness = ('completeness_score', 'mean'),
    coherence    = ('coherence_score',    'mean'),
    fidelity     = ('fidelity',           'mean'),
).round(3)
print('\nQuality by Protocol x Domain:')
print(quality_domain.to_string())

## 16. RQ3 — Failure Propagation & Error Detection

In [ ]:
failures = df[df['failure_mode'] != 'NONE']

fail_summary = failures.groupby(['protocol', 'failure_mode']).agg(
    detection_rate   = ('error_detected',         'mean'),
    propagation_rate = ('error_propagation_rate', 'mean'),
    recovery_rate    = ('recovery_success',        'mean'),
    degradation_rate = ('report_degraded',         'mean'),
).round(3)

print('RQ3: Failure Propagation — Protocol x Failure Mode')
print(fail_summary.to_string())

# ── Two-way ANOVA ─────────────────────────────────────────────────────────
try:
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    model = ols('error_propagation_rate ~ C(protocol) + C(failure_mode) + C(protocol):C(failure_mode)',
                data=failures).fit()
    print('\nTwo-way ANOVA (error_propagation_rate ~ protocol * failure_mode):')
    print(sm.stats.anova_lm(model, typ=2).round(4))
except Exception as e:
    print(f'ANOVA error: {e}')

# ── Reliability by Domain ────────────────────────────────────────────────
fail_domain = failures.groupby(['protocol', 'domain']).agg(
    detection   = ('error_detected',         'mean'),
    propagation = ('error_propagation_rate', 'mean'),
).round(3)
print('\nReliability by Protocol x Domain:')
print(fail_domain.to_string())

## 17. Bootstrap 95% Confidence Intervals

In [ ]:
def bootstrap_ci(data: np.ndarray, n_boot=2000, alpha=0.05) -> Tuple[float, float]:
    boot = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(n_boot)]
    return round(np.percentile(boot, 100*alpha/2), 3), round(np.percentile(boot, 100*(1-alpha/2)), 3)

print('=== Efficiency: Token usage (baseline) ===')
for p in ['NL','MARKDOWN','JSON','SHARED']:
    d = baseline[baseline['protocol']==p]['total_tokens'].values
    if len(d) > 0:
        lo, hi = bootstrap_ci(d)
        print(f'  {p:<10} mean={np.mean(d):7.1f}  95% CI [{lo}, {hi}]')

print('\n=== Quality: Accuracy score (baseline) ===')
for p in ['NL','MARKDOWN','JSON','SHARED']:
    d = baseline[baseline['protocol']==p]['accuracy_score'].values
    if len(d) > 0:
        lo, hi = bootstrap_ci(d)
        print(f'  {p:<10} mean={np.mean(d):5.2f}  95% CI [{lo}, {hi}]')

print('\n=== Quality: Fidelity (baseline) ===')
for p in ['NL','MARKDOWN','JSON','SHARED']:
    d = baseline[baseline['protocol']==p]['fidelity'].values
    if len(d) > 0:
        lo, hi = bootstrap_ci(d)
        print(f'  {p:<10} mean={np.mean(d):5.3f}  95% CI [{lo}, {hi}]')

print('\n=== Reliability: Error detection rate (failure runs) ===')
for p in ['NL','MARKDOWN','JSON','SHARED']:
    d = failures[failures['protocol']==p]['error_detected'].astype(float).values
    if len(d) > 0:
        lo, hi = bootstrap_ci(d)
        print(f'  {p:<10} mean={np.mean(d):.3f}  95% CI [{lo}, {hi}]')

## 18. Multi-Factor ANOVA (Protocol × Domain × Failure)

In [ ]:
try:
    import statsmodels.api as sm
    from statsmodels.formula.api import ols

    # ── Token consumption ─────────────────────────────────────────────────
    m1 = ols('total_tokens ~ C(protocol) * C(domain) * C(failure_mode)', data=df).fit()
    print('=== ANOVA: total_tokens ~ protocol * domain * failure_mode ===')
    print(sm.stats.anova_lm(m1, typ=2).round(4).to_string())

    # ── Accuracy score ────────────────────────────────────────────────────
    m2 = ols('accuracy_score ~ C(protocol) * C(domain) * C(failure_mode)', data=df).fit()
    print('\n=== ANOVA: accuracy_score ~ protocol * domain * failure_mode ===')
    print(sm.stats.anova_lm(m2, typ=2).round(4).to_string())

    # ── Error propagation ─────────────────────────────────────────────────
    m3 = ols('error_propagation_rate ~ C(protocol) * C(domain) * C(failure_mode)', data=df).fit()
    print('\n=== ANOVA: error_propagation_rate ~ protocol * domain * failure_mode ===')
    print(sm.stats.anova_lm(m3, typ=2).round(4).to_string())

except Exception as e:
    print(f'Multi-factor ANOVA error: {e}')

## 19. Markov Chain — Error Propagation Model

In [ ]:
def build_markov(pdata: pd.DataFrame) -> np.ndarray:
    """States: 0=OK, 1=ERROR_INJECTED, 2=ERROR_DETECTED, 3=ERROR_PROPAGATED"""
    n        = len(pdata)
    p_inject = (pdata['failure_mode'] != 'NONE').sum() / n if n > 0 else 0
    p_detect = pdata['error_detected'].mean()
    p_recov  = pdata['recovery_success'].mean()
    return np.array([
        [1-p_inject, p_inject,   0,          0          ],
        [0,          0,          p_detect,   1-p_detect  ],
        [p_recov,    1-p_recov,  0,          0          ],
        [0,          0,          0,          1          ],
    ])

states = ['OK','ERR_INJECT','ERR_DETECT','ERR_PROP']
print('Markov Transition Matrices\n')
for p in ['NL','MARKDOWN','JSON','SHARED']:
    sub = df[df['protocol']==p]
    if len(sub) == 0:
        continue
    T  = build_markov(sub)
    sv = np.array([1,0,0,0], dtype=float)
    for _ in range(10):
        sv = sv @ T
    print(f'[{p}]')
    hdr = f'{"":>14}' + ''.join(f'{s:>14}' for s in states)
    print(hdr)
    for i, lbl in enumerate(states):
        print(f'{lbl:>14}' + ''.join(f'{T[i,j]:>14.3f}' for j in range(4)))
    print(f'  P(absorbed -> ERR_PROP after 10 steps): {sv[3]:.3f}\n')

## 20. Final Summary Table — All Three Dimensions

In [ ]:
rows = []
for p in ['NL','MARKDOWN','JSON','SHARED']:
    b = df[(df['protocol']==p) & (df['failure_mode']=='NONE')]
    f = df[(df['protocol']==p) & (df['failure_mode']!='NONE')]
    if len(b) == 0:
        continue
    rows.append({
        'Protocol':            p,
        # Efficiency
        'Tokens (baseline)':   round(b['total_tokens'].mean(), 1),
        'Latency ms (base)':   round(b['total_latency_ms'].mean(), 1),
        # Quality
        'Accuracy':            round(b['accuracy_score'].mean(), 2),
        'Completeness':        round(b['completeness_score'].mean(), 2),
        'Coherence':           round(b['coherence_score'].mean(), 2),
        'Fidelity':            round(b['fidelity'].mean(), 3),
        # Reliability
        'Error Detection':     round(f['error_detected'].mean(), 3) if len(f) > 0 else None,
        'Error Propagation':   round(f['error_propagation_rate'].mean(), 3) if len(f) > 0 else None,
        'Recovery Rate':       round(f['recovery_success'].mean(), 3) if len(f) > 0 else None,
    })

summary = pd.DataFrame(rows).set_index('Protocol')
print('FINAL SUMMARY — Efficiency / Quality / Reliability')
print(summary.to_string())
summary.to_csv('results_summary.csv')
print('\nSaved: results_summary.csv')
print('\nGuide:')
print('  Efficiency:   Tokens/Latency → lower is better')
print('  Quality:      Accuracy/Completeness/Coherence/Fidelity → higher is better')
print('  Reliability:  Detection/Recovery → higher is better | Propagation → lower is better')

## 21. Sample Report Outputs

In [ ]:
print('Sample report snippets (first per condition)\n')
for p in ['NL','MARKDOWN','JSON','SHARED']:
    for d in ['MATH','READING','NEWS']:
        row = df[(df['protocol']==p) & (df['domain']==d) & (df['failure_mode']=='NONE')]
        if len(row) > 0:
            r = row.iloc[0]
            print(f'[{p}/{d}] acc={r["accuracy_score"]:.1f} comp={r["completeness_score"]:.1f} '
                  f'coh={r["coherence_score"]:.1f} fid={r["fidelity"]:.3f}')
            print(f'  {r["report_snippet"][:150]}')
            print()